In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install pytorch-lightning pytorch-forecasting -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 682.3 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 3.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 11.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 10.1 MB/s eta 0:00:00


In [2]:
import torch
import pytorch_forecasting

print("PyTorch:", torch.__version__)
print("TFT libraries installed successfully")

PyTorch: 2.10.0+cpu
TFT libraries installed successfully


In [9]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/aditio2311
/kaggle/input/datasets/aditio2311/hacathon-2-0
    calendar.csv
    sample_submission.csv
    sell_prices.csv
    sales_train_validation.csv
    sales_train_evaluation.csv


In [10]:
import pandas as pd

sales = pd.read_csv('/kaggle/input/datasets/aditio2311/hacathon-2-0/sales_train_validation.csv')
calendar = pd.read_csv('/kaggle/input/datasets/aditio2311/hacathon-2-0/calendar.csv')

print(sales.shape)

(30490, 1919)


In [11]:
sales_small = sales.head(50)

print(sales_small.shape)

(50, 1919)


In [12]:
import torch
import pytorch_forecasting

print("TFT libraries installed successfully")

TFT libraries installed successfully


In [13]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/aditio2311
/kaggle/input/datasets/aditio2311/hacathon-2-0


In [14]:
import pandas as pd

sales = pd.read_csv('/kaggle/input/datasets/aditio2311/hacathon-2-0/sales_train_validation.csv')
calendar = pd.read_csv('/kaggle/input/datasets/aditio2311/hacathon-2-0/calendar.csv')

print(sales.shape)

(30490, 1919)


In [16]:
sales_small = sales.head(20)

id_cols = [
    'id',
    'item_id',
    'dept_id',
    'cat_id',
    'store_id',
    'state_id'
]

sales_long = sales_small.melt(
    id_vars=id_cols,
    var_name='d',
    value_name='sales'
)

sales_long = sales_long.merge(
    calendar[['d', 'date']],
    on='d',
    how='left'
)

sales_long['date'] = pd.to_datetime(sales_long['date'])

In [17]:
sales_long = sales_long.sort_values(['id', 'date'])

sales_long['time_idx'] = (
    sales_long.groupby('id')
    .cumcount()
)

print(sales_long.shape)
sales_long.head()

(38260, 10)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,time_idx
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,0
20,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0,2011-01-30,1
40,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0,2011-01-31,2
60,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_4,0,2011-02-01,3
80,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_5,0,2011-02-02,4


In [18]:
print(sales_long.shape)
sales_long.head()

(38260, 10)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,time_idx
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,0
20,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0,2011-01-30,1
40,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0,2011-01-31,2
60,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_4,0,2011-02-01,3
80,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_5,0,2011-02-02,4


In [19]:
sales_long['month'] = sales_long['date'].dt.month.astype(str)
sales_long['dayofweek'] = sales_long['date'].dt.dayofweek.astype(str)

In [20]:
max_time_idx = sales_long['time_idx'].max()

training_cutoff = max_time_idx - 28

print("Max time_idx:", max_time_idx)
print("Training cutoff:", training_cutoff)

Max time_idx: 1912
Training cutoff: 1884


In [21]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

max_encoder_length = 60
max_prediction_length = 28

training = TimeSeriesDataSet(
    sales_long[sales_long.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="sales",
    group_ids=["id"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    static_categoricals=[
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],

    time_varying_known_categoricals=[
        "month",
        "dayofweek"
    ],

    time_varying_unknown_reals=[
        "sales"
    ],

    target_normalizer=GroupNormalizer(
        groups=["id"]
    )
)

In [22]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    sales_long,
    predict=True,
    stop_randomization=True
)

print("Dataset created successfully")

Dataset created successfully


In [23]:
batch_size = 64

train_dataloader = training.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=0
)

val_dataloader = validation.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0
)

print("Dataloaders created")

Dataloaders created


In [24]:
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,
    loss=QuantileLoss(),
)

print("TFT model created")

TFT model created


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [25]:
print(f"Number of parameters: {tft.size()/1e3:.1f}k")

Number of parameters: 15.7k


In [26]:
import lightning.pytorch as pl

trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    logger=False
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    291 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │     16 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │    425 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │    783 │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │    180 │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 15.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 15.7 K                                                                                               
Total estimated model params size (MB): 0.063                                                                      
Modules in train mode: 215                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=5` reached.


In [27]:
predictions = tft.predict(
    val_dataloader,
    return_x=False
)

print(predictions.shape)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
2026-06-24 13:39:52.400200: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782308392.654144      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782308392.725666      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to

torch.Size([20, 28])


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


In [28]:
import pandas as pd
import numpy as np

pred_df = pd.DataFrame({
    "tft_pred": np.array(predictions).flatten()
})

pred_df.to_csv("tft_predictions.csv", index=False)

pred_df.head()

/tmp/ipykernel_58/2052635548.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  "tft_pred": np.array(predictions).flatten()


,tft_pred
0,0.028198
1,0.047835
2,-0.001152
3,0.029530
4,0.076011


In [29]:
print(pred_df.describe())

         tft_pred
count  560.000000
mean     0.811521
std      1.211849
min     -0.073945
25%      0.029197
50%      0.330184
75%      0.864217
max      5.213267


In [30]:
pred_df.to_csv("tft_predictions.csv", index=False)

print("TFT predictions saved")

TFT predictions saved
